In [3]:
import cProfile
import pandas as pd
from src.recovery_model import RecoveryModel
import os

pd.set_option("multi_sparse", False)
pd.set_option("display.float_format", "{:.2f}".format)

### Expanding data size to factor

In [33]:
#importing from Toy_WEEE_v2 and expanding data size
folder = "Toy_WEEE_v2"
path = f"data/{folder}/"

#factor to expand multiple input flows
nfactor = 20 
exp_flows = range(nfactor)

#original
inputs = pd.read_csv(path + "inputs.csv")
composition = pd.read_csv(path + "composition.csv")
TCs = pd.read_csv(path + "TCs.csv")
metadata = pd.read_csv(path + "metadata.csv")

#expanding flows
#for TC just the initial flow, rest all same
TCs_init_st = TCs[TCs['input_flow']=='WEEE_generatedDedicated']
TCs_late_st = TCs[TCs['input_flow']!='WEEE_generatedDedicated']

# List to store expanded dfs
dfs_inputs = []
dfs_composition = []
dfs_TCs = []
dfs_flows = []

for i in exp_flows:
    #inputs
    df_tmp = inputs.copy()
    df_tmp['Stock/Flow ID'] = df_tmp['Stock/Flow ID'] + str(i)
    dfs_inputs.append(df_tmp)

    #cmp
    df_tmp = composition.copy()
    df_tmp['flow'] = df_tmp['flow'] + str(i)
    dfs_composition.append(df_tmp)

    #TCs
    df_tmp = TCs_init_st.copy()
    df_tmp['input_flow'] = df_tmp['input_flow'] + str(i)
    dfs_TCs.append(df_tmp)

    #new flows
    dfs_flows.append('WEEE_generatedDedicated'+ str(i))

#New + old flows of metadata 
dfs_flows = pd.DataFrame(data = {'flow': dfs_flows + list(pd.unique(metadata['flow'].dropna()))})

#expanded dfs
inputs_exp = pd.concat(dfs_inputs, ignore_index=True)
composition_exp = pd.concat(dfs_composition, ignore_index=True)
TCs_exp = pd.concat(dfs_TCs, ignore_index=True)
metadata_exp = pd.concat([metadata.drop(columns=['flow']).reset_index(), dfs_flows], axis=1)
metadata_exp = metadata_exp[list(metadata.columns)] #metadata columns order

#Add late stage TCs
TCs_exp = pd.concat([TCs_exp, TCs_late_st], ignore_index=True)

#Save to new folder
folder_out = "test_size_factor" + str(nfactor)
path_out = f"data/{folder_out}/"

if not os.path.exists(path_out):
    os.makedirs(path_out)



inputs_exp.to_csv(path_out + 'inputs.csv', index=False)
composition_exp.to_csv(path_out + 'composition.csv', index=False)
TCs_exp.to_csv(path_out + 'TCs.csv', index=False)
metadata_exp.to_csv(path_out + 'metadata.csv', index=False)






# 1 Testing size

### 1.1 Size factor 5

In [8]:
#Select a folder for the data to be used
folder = "test_size_factor5"  # test_1  test_2  Toy_WEEE_v2 test_size_mult_inputs test_fewer_layers test_extra_layers test_skip_layers
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
layer_4 = "element"
#layer_5 = "glayer"

In [9]:
#for normal layers
layer_names = (layer_0, layer_1, layer_2, layer_3, layer_4)
all_symbols = {layer_1: "P*", layer_2: "C*", layer_3: "M*", layer_4: "E*"}

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Layer 4": layer_4,
        "Value": "data",
        "Year": "year",
        "Scenario": "scenario",
        "Location": "region",
        "parameterCode": "parameterCode",
        "UoM": "unit",
    },
    "parameterCode": {
        layer_2: "c-p",
        layer_3: "m-c",
        layer_4: "e-m"
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Year": "year",
        "Unit": "unit",
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
        "process": "process",
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        layer_4: "E*",
    },
}

In [10]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

In [14]:
model.dims

(24, 8, 83, 31, 18)

In [15]:
model.size

8892288

In [16]:
model.flows_eqs

,WEEE_2RM_chem2Smelter_other,WEEE_2RM_mechRec1Smelter_other,WEEE_2RM_mechRec2Smelter_other,WEEE_2RM_thermal2Smelter_other,WEEE_categ_mechRec1,WEEE_categ_mechRec2,WEEE_chem_thermal,WEEE_collected,WEEE_generatedComplementaryExported,WEEE_generatedComplementaryMSW,...,WEEE_generatedDedicated1,WEEE_generatedDedicated2,WEEE_generatedDedicated3,WEEE_generatedDedicated4,WEEE_mechRec1_mechRec2,WEEE_mechRec2_chem,WEEE_mechRec2_thermal,WEEE_thermal_chem,WEEE_wasteBinIncineration,WEEE_wasteBinLandfill
process,,,,,,,,,,,,,,,,,,,,,
WEEE_complementary_collection(MSW),0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,-1,-1
WEEE_generated,0,0,0,0,0,0,0,-1,-1,-1,...,1,1,1,1,0,0,0,0,0,0
categorizing,0,0,0,0,-1,-1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
chemical process,-1,0,0,0,0,0,-1,0,0,0,...,0,0,0,0,0,1,0,1,0,0
dismantling,0,-1,0,0,1,0,0,0,0,0,...,0,0,0,0,-1,0,0,0,0,0
shredding,0,0,-1,0,0,1,0,0,0,0,...,0,0,0,0,1,-1,-1,0,0,0
thermal process,0,0,0,-1,0,0,1,0,0,0,...,0,0,0,0,0,0,1,-1,0,0


In [17]:
list(enumerate(model.sub_systems))

[(0, {'WEEE_generated'}),
 (1, {'WEEE_complementary_collection(MSW)'}),
 (2, {'categorizing'}),
 (3, {'dismantling'}),
 (4, {'shredding'}),
 (5, {'chemical process', 'thermal process'})]

In [18]:
print(model)

{   'Layer 1': {   'P*': 7,
                   'WEEE_Cat1': 0,
                   'WEEE_Cat2': 1,
                   'WEEE_Cat3': 2,
                   'WEEE_Cat4a': 3,
                   'WEEE_Cat4b': 4,
                   'WEEE_Cat5': 5,
                   'WEEE_Cat6': 6},
    'Layer 2': {   'C*': 82,
                   'ComponentShadowWEEE': 0,
                   'ConductivitySensor': 1,
                   'LED': 2,
                   'LidarSensor (&lt;15cm)': 3,
                   'MagnetSensor': 4,
                   'PCBWEEECat': 5,
                   'PCBunspecified': 6,
                   'PVBacksheet': 7,
                   'PVEncapsulationEVA': 8,
                   'PVFrameAl': 9,
                   'PVSolarGlass': 10,
                   'PVcellSi': 11,
                   'WEEEEvaporator': 12,
                   'brush': 13,
                   'cableUnspecified': 14,
                   'cableWOPlug': 15,
                   'cableWPlug': 16,
                   'cableWithConne

In [19]:
model.lneqs

<8892288x8892288 sparse matrix of type '<class 'numpy.float64'>'
	with 9825360 stored elements in Compressed Sparse Row format>

In [20]:
model.y

<8892288x1 sparse array of type '<class 'numpy.int64'>'
	with 35 stored elements in Compressed Sparse Column format>

In [21]:
solution = model.solve(aggregate=True, pivot=True)
solution.fillna("")

,flow,WEEE_categ_mechRec1,WEEE_categ_mechRec2,WEEE_collected,WEEE_generatedComplementaryExported,WEEE_generatedComplementaryMSW,WEEE_generatedComplementaryMetalScrap,WEEE_generatedComplementaryUndocumented,WEEE_generatedDedicated0,WEEE_generatedDedicated1,WEEE_generatedDedicated2,...,WEEE_wasteBinLandfill,WEEE_2RM_mechRec1Smelter_other,WEEE_mechRec1_mechRec2,WEEE_2RM_mechRec2Smelter_other,WEEE_mechRec2_chem,WEEE_mechRec2_thermal,WEEE_2RM_chem2Smelter_other,WEEE_2RM_thermal2Smelter_other,WEEE_chem_thermal,WEEE_thermal_chem
layer,key,,,,,,,,,,,,,,,,,,,,,
product,WEEE_Cat1,1612592.45,2324030.29,4742918.96,677559.85,4658223.98,1778594.61,4150054.09,1693899.63,1693899.63,1693899.63,...,1676960.63,,,,,,,,,
product,WEEE_Cat2,499525.58,83254.26,925047.38,616698.25,2672359.09,1644528.67,2364009.96,1027830.42,1027830.42,1027830.42,...,454301.04,,,,,,,,,
product,WEEE_Cat3,37466.63,25289.97,93666.56,104686.16,314058.48,82646.97,126725.35,110195.96,110195.96,110195.96,...,91076.96,,,,,,,,,
product,WEEE_Cat4a,878888.89,2636666.66,6277777.75,2092592.58,8602880.62,10462962.92,8602880.62,4650205.74,4650205.74,4650205.74,...,2494835.38,,,,,,,,,
product,WEEE_Cat4b,21739.54,149459.35,271744.28,242366.52,212988.76,403944.20,183611.00,146888.80,146888.80,146888.80,...,44727.64,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
element,Pd,20689.13,38451.87,92931.38,106490.72,160820.17,182351.79,141771.16,81649.37,81649.37,81649.37,...,46857.79,1515.16,1068.87,744.78,,,,,,
element,Sc,37642.88,74734.88,184113.20,254194.44,353783.34,403377.25,289634.17,175588.23,175588.23,175588.23,...,105443.44,3185.29,2137.97,728.54,,,,,,
element,Sm,151148.47,287395.26,655505.88,611702.39,1020930.02,1071805.00,881372.70,499460.71,499460.71,499460.71,...,316540.46,4917.86,4917.12,18377.51,22.82,15.04,,,,


### Size factor 20

In [34]:
#Select a folder for the data to be used
folder = "test_size_factor20"  # test_1  test_2  Toy_WEEE_v2 test_size_factor5 test_fewer_layers test_extra_layers test_skip_layers
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
layer_4 = "element"
#layer_5 = "glayer"

In [35]:
#for normal layers
layer_names = (layer_0, layer_1, layer_2, layer_3, layer_4)
all_symbols = {layer_1: "P*", layer_2: "C*", layer_3: "M*", layer_4: "E*"}

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Layer 4": layer_4,
        "Value": "data",
        "Year": "year",
        "Scenario": "scenario",
        "Location": "region",
        "parameterCode": "parameterCode",
        "UoM": "unit",
    },
    "parameterCode": {
        layer_2: "c-p",
        layer_3: "m-c",
        layer_4: "e-m"
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Year": "year",
        "Unit": "unit",
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
        "process": "process",
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        layer_4: "E*",
    },
}

In [36]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

In [37]:
model.dims

(39, 8, 83, 31, 18)

In [38]:
model.size

14449968

In [39]:
model.flows_eqs

,WEEE_2RM_chem2Smelter_other,WEEE_2RM_mechRec1Smelter_other,WEEE_2RM_mechRec2Smelter_other,WEEE_2RM_thermal2Smelter_other,WEEE_categ_mechRec1,WEEE_categ_mechRec2,WEEE_chem_thermal,WEEE_collected,WEEE_generatedComplementaryExported,WEEE_generatedComplementaryMSW,...,WEEE_generatedDedicated6,WEEE_generatedDedicated7,WEEE_generatedDedicated8,WEEE_generatedDedicated9,WEEE_mechRec1_mechRec2,WEEE_mechRec2_chem,WEEE_mechRec2_thermal,WEEE_thermal_chem,WEEE_wasteBinIncineration,WEEE_wasteBinLandfill
process,,,,,,,,,,,,,,,,,,,,,
WEEE_complementary_collection(MSW),0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,-1,-1
WEEE_generated,0,0,0,0,0,0,0,-1,-1,-1,...,1,1,1,1,0,0,0,0,0,0
categorizing,0,0,0,0,-1,-1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
chemical process,-1,0,0,0,0,0,-1,0,0,0,...,0,0,0,0,0,1,0,1,0,0
dismantling,0,-1,0,0,1,0,0,0,0,0,...,0,0,0,0,-1,0,0,0,0,0
shredding,0,0,-1,0,0,1,0,0,0,0,...,0,0,0,0,1,-1,-1,0,0,0
thermal process,0,0,0,-1,0,0,1,0,0,0,...,0,0,0,0,0,0,1,-1,0,0


In [40]:
list(enumerate(model.sub_systems))

[(0, {'WEEE_generated'}),
 (1, {'WEEE_complementary_collection(MSW)'}),
 (2, {'categorizing'}),
 (3, {'dismantling'}),
 (4, {'shredding'}),
 (5, {'chemical process', 'thermal process'})]

In [41]:
print(model)

{   'Layer 1': {   'P*': 7,
                   'WEEE_Cat1': 0,
                   'WEEE_Cat2': 1,
                   'WEEE_Cat3': 2,
                   'WEEE_Cat4a': 3,
                   'WEEE_Cat4b': 4,
                   'WEEE_Cat5': 5,
                   'WEEE_Cat6': 6},
    'Layer 2': {   'C*': 82,
                   'ComponentShadowWEEE': 0,
                   'ConductivitySensor': 1,
                   'LED': 2,
                   'LidarSensor (&lt;15cm)': 3,
                   'MagnetSensor': 4,
                   'PCBWEEECat': 5,
                   'PCBunspecified': 6,
                   'PVBacksheet': 7,
                   'PVEncapsulationEVA': 8,
                   'PVFrameAl': 9,
                   'PVSolarGlass': 10,
                   'PVcellSi': 11,
                   'WEEEEvaporator': 12,
                   'brush': 13,
                   'cableUnspecified': 14,
                   'cableWOPlug': 15,
                   'cableWPlug': 16,
                   'cableWithConne

In [42]:
model.lneqs

<14449968x14449968 sparse matrix of type '<class 'numpy.float64'>'
	with 34260540 stored elements in Compressed Sparse Row format>

In [43]:
model.y

<14449968x1 sparse array of type '<class 'numpy.int64'>'
	with 140 stored elements in Compressed Sparse Column format>

In [44]:
solution = model.solve(aggregate=True, pivot=True)
solution.fillna("")

RuntimeError: SUPERLU_MALLOC fails for buf in intCalloc() at line 172 in file ../scipy/sparse/linalg/_dsolve/SuperLU/SRC/memory.c


# 2 Testing layers change

### 2.1 Fewer layers

In [73]:
#Select a folder for the data to be used
folder = "test_fewer_layers"  # test_1  test_2  Toy_WEEE_v2 test_size_mult_inputs test_fewer_layers test_extra_layers test_skip_layers
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
#layer_4 = "element"
#layer_5 = "glayer"

In [74]:
#for test_fewer_layers
layer_names = (layer_0, layer_1, layer_2, layer_3)
all_symbols = {layer_1: "P*", layer_2: "C*", layer_3: "M*"}

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Value": "data",
        "Year": "year",
        "Scenario": "scenario",
        "Location": "region",
        "parameterCode": "parameterCode",
        "UoM": "unit",
    },
    "parameterCode": {
        layer_2: "c-p",
        layer_3: "m-c",
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Year": "year",
        "Unit": "unit",
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
        "process": "process",
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
    },
}

In [75]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

In [76]:
model.dims

(7, 3, 4, 3)

In [77]:
model.size

252

In [78]:
model.flows_eqs

,F1,F2,F3,F4,F5,F6,F7
process,,,,,,,
T1,1,-1,-1,0,0,1,0
T2,0,1,0,-1,-1,0,0
T4,0,0,1,1,0,-1,-1


In [79]:
list(enumerate(model.sub_systems))

[(0, {'T1', 'T2', 'T4'})]

In [80]:
print(model)

{   'Layer 1': {'P*': 2, 'P1': 0, 'P2': 1},
    'Layer 2': {'C*': 3, 'C1': 0, 'C2': 1, 'C3': 2},
    'Layer 3': {'M*': 2, 'M1': 0, 'M2': 1},
    'Stock/Flow ID': {   'F1': 0,
                         'F2': 1,
                         'F3': 2,
                         'F4': 3,
                         'F5': 4,
                         'F6': 5,
                         'F7': 6},
    'Substance_main_parent': {'P*': 2, 'P1': 0, 'P2': 1},
    'Unit': {'Mg': 0},
    'component': {'C*': 3, 'C1': 0, 'C2': 1, 'C3': 2},
    'flow': {'F1': 0, 'F2': 1, 'F3': 2, 'F4': 3, 'F5': 4, 'F6': 5, 'F7': 6},
    'material': {'M*': 2, 'M1': 0, 'M2': 1},
    'parameterCode': {'c-p': 0, 'm-c': 1},
    'process': {'T1': 0, 'T2': 1, 'T4': 2},
    'product': {'P*': 2, 'P1': 0, 'P2': 1}}


In [81]:
model.lneqs

<252x252 sparse matrix of type '<class 'numpy.float64'>'
	with 135 stored elements in Compressed Sparse Row format>

In [82]:
model.y

<252x1 sparse array of type '<class 'numpy.int64'>'
	with 2 stored elements in Compressed Sparse Column format>

In [83]:
solution = model.solve(aggregate=True, pivot=True)
solution.fillna("")

,flow,F1,F2,F3,F4,F6,F5,F7
layer,key,,,,,,,
product,P1,1000.00,,,,,,
product,P2,700.00,,,,,,
component,C1,551.00,86.58,,,,,
component,C2,639.00,51.57,39.40,10.31,17.32,,
component,C3,510.00,,13.04,,1.83,,
material,M1,920.43,84.53,35.28,7.90,13.72,21.51,4.56
material,M2,779.57,53.62,17.16,2.41,5.42,19.33,4.99


In [86]:
solution = model.solve(aggregate=False, pivot=True)
solution.fillna("")

flow,product,component,material,F1,F2,F3,F4,F5,F6,F7
0,P1,,,1000.00,,,,,,
1,P1,C1,,250.00,62.50,,,,,
2,P1,C1,M1,130.00,32.50,,,12.68,,
3,P1,C1,M2,120.00,30.00,,,9.60,,
4,P1,C2,,590.00,27.76,26.86,5.55,,11.24,
5,P1,C2,M1,389.40,18.32,17.73,3.66,1.83,7.42,2.08
6,P1,C2,M2,200.60,9.44,9.13,1.89,4.72,3.82,2.04
7,P1,C3,,160.00,,13.04,,,1.83,
8,P1,C3,M1,78.40,,6.39,,,0.89,0.83
9,P1,C3,M2,81.60,,6.65,,,0.93,2.59


### 2.1b Fewer layers no loop

In [92]:
#Select a folder for the data to be used
folder = "test_fewer_layers_no_loop"  # test_1  test_2  Toy_WEEE_v2 test_size_mult_inputs test_fewer_layers test_extra_layers test_skip_layers
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
#layer_4 = "element"
#layer_5 = "glayer"

In [89]:
#for test_fewer_layers
layer_names = (layer_0, layer_1, layer_2, layer_3)
all_symbols = {layer_1: "P*", layer_2: "C*", layer_3: "M*"}

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Value": "data",
        "Year": "year",
        "Scenario": "scenario",
        "Location": "region",
        "parameterCode": "parameterCode",
        "UoM": "unit",
    },
    "parameterCode": {
        layer_2: "c-p",
        layer_3: "m-c",
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Year": "year",
        "Unit": "unit",
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
        "process": "process",
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
    },
}

In [93]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

In [94]:
solution = model.solve(aggregate=False, pivot=True)
solution.fillna("")

flow,product,component,material,F1,F2,F3,F4,F5,F6,F7
0,P1,,,1000.00,,,,,,
1,P1,C1,,250.00,62.50,,,,,
2,P1,C1,M1,130.00,32.50,,,12.68,,
3,P1,C1,M2,120.00,30.00,,,9.60,,
4,P1,C2,,590.00,23.60,23.60,4.72,,9.82,
5,P1,C2,M1,389.40,15.58,15.58,3.12,1.56,6.48,1.81
6,P1,C2,M2,200.60,8.02,8.02,1.60,4.01,3.34,1.78
7,P1,C3,,160.00,,12.80,,,1.79,
8,P1,C3,M1,78.40,,6.27,,,0.88,0.82
9,P1,C3,M2,81.60,,6.53,,,0.91,2.55


### 2.2 Extra layers

In [57]:
#Select a folder for the data to be used
folder = "test_extra_layers"  # test_1  test_2  Toy_WEEE_v2 test_size_mult_inputs test_fewer_layers test_extra_layers test_skip_layers
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
layer_4 = "element"
layer_5 = "glayer"

In [59]:
#for extra layers
layer_names = (layer_0, layer_1, layer_2, layer_3, layer_4, layer_5)
all_symbols = {layer_1: "P*", layer_2: "C*", layer_3: "M*", layer_4: "E*", layer_5: "G*"}

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Layer 4": layer_4,
        "Layer 5": layer_5,
        "Value": "data",
        "Year": "year",
        "Scenario": "scenario",
        "Location": "region",
        "parameterCode": "parameterCode",
        "UoM": "unit",
    },
    "parameterCode": {
        layer_2: "c-p",
        layer_3: "m-c",
        layer_4: "e-m",
        layer_5: "g-e",
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Year": "year",
        "Unit": "unit",
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
        "process": "process",
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        layer_4: "E*",
        layer_5: "G*",
    },
}

In [60]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

In [61]:
model.dims

(9, 3, 4, 3, 3, 3)

In [62]:
model.size

2916

In [63]:
model.flows_eqs

,F1,F2,F3,F4,F5,F6,F7,F8,F9
process,,,,,,,,,
T1,1,-1,-1,0,0,1,0,0,0
T2,0,1,0,-1,-1,0,0,0,0
T3,0,0,0,0,1,0,1,-1,0
T4,0,0,1,1,0,-1,-1,0,0
T5,0,0,0,0,0,0,0,1,-1


In [64]:
list(enumerate(model.sub_systems))

[(0, {'T1', 'T2', 'T4'}), (1, {'T3'}), (2, {'T5'})]

In [ ]:
print(model)

{   'Layer 1': {'P*': 2, 'P1': 0, 'P2': 1},
    'Layer 2': {'C*': 3, 'C1': 0, 'C2': 1, 'C3': 2},
    'Layer 3': {'M*': 2, 'M1': 0, 'M2': 1},
    'Stock/Flow ID': {   'F1': 0,
                         'F2': 1,
                         'F3': 2,
                         'F4': 3,
                         'F5': 4,
                         'F6': 5,
                         'F7': 6},
    'Substance_main_parent': {'P*': 2, 'P1': 0, 'P2': 1},
    'Unit': {'Mg': 0},
    'component': {'C*': 3, 'C1': 0, 'C2': 1, 'C3': 2},
    'flow': {'F1': 0, 'F2': 1, 'F3': 2, 'F4': 3, 'F5': 4, 'F6': 5, 'F7': 6},
    'material': {'M*': 2, 'M1': 0, 'M2': 1},
    'parameterCode': {'c-p': 0, 'm-c': 1},
    'process': {'T1': 0, 'T2': 1, 'T4': 2},
    'product': {'P*': 2, 'P1': 0, 'P2': 1}}


In [65]:
model.lneqs

<2916x2916 sparse matrix of type '<class 'numpy.float64'>'
	with 1407 stored elements in Compressed Sparse Row format>

In [66]:
model.y

<2916x1 sparse array of type '<class 'numpy.int64'>'
	with 2 stored elements in Compressed Sparse Column format>

In [67]:
solution = model.solve(aggregate=True, pivot=True)
solution.fillna("")

,flow,F1,F2,F3,F4,F6,F5,F7,F8,F9
layer,key,,,,,,,,,
product,P1,1000.00,,,,,,,,
product,P2,700.00,,,,,,,,
component,C1,551.00,86.58,,,,,,,
component,C2,639.00,51.57,39.40,10.31,17.32,,,,
component,C3,510.00,,13.04,,1.83,,,,
material,M1,920.43,84.53,35.28,7.90,13.72,21.51,4.56,,
material,M2,779.57,53.62,17.16,2.41,5.42,19.33,4.99,,
element,E1,676.76,79.40,14.76,3.39,5.66,24.33,2.42,2.32,
element,E2,1023.24,58.75,37.68,6.93,13.48,16.51,7.14,1.19,


### 2.3 Skip layers

In [68]:
#Select a folder for the data to be used
folder = "test_skip_layers"  # test_1  test_2  Toy_WEEE_v2 test_size_mult_inputs test_fewer_layers test_extra_layers test_skip_layers
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
layer_4 = "element"
#layer_5 = "glayer"

In [72]:
#for skip layers

layer_names = (layer_0, layer_1, layer_2, layer_3, layer_4)
all_symbols = {layer_1: "P*", layer_2: "C*", layer_3: "M*", layer_4: "E*"}

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Layer 4": layer_4,
        "Value": "data",
        "Year": "year",
        "Scenario": "scenario",
        "Location": "region",
        "parameterCode": "parameterCode",
        "UoM": "unit",
    },
    "parameterCode": {
        layer_2: "c-p",
        layer_3: "m-p",
        layer_4: "e-m", #"e-c"
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Year": "year",
        "Unit": "unit",
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
        "process": "process",
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        layer_4: "E*",
    },
}

In [71]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

#### it would not get "e-c" in parameter code, require change in method read_composition in class RecoveryModel

---


# Recovery model


In [70]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

---

# Metadata


In [18]:
model.dims

(4, 3, 4, 3, 3)

In [19]:
model.size

432

In [25]:
model.flows_eqs

,F1,F2,F3,F4,F5,F6,F7,F8,F9
process,,,,,,,,,
T1,1,-1,-1,0,0,1,0,0,0
T2,0,1,0,-1,-1,0,0,0,0
T3,0,0,0,0,1,0,1,-1,0
T4,0,0,1,1,0,-1,-1,0,0
T5,0,0,0,0,0,0,0,1,-1


In [20]:
list(enumerate(model.sub_systems))

[(0, {'T1'}), (1, {'T2'})]

In [21]:
print(model)

{   'Layer 1': {'P*': 2, 'P1': 0, 'P2': 1},
    'Layer 2': {'C*': 3, 'C1': 0, 'C2': 1, 'C3': 2},
    'Layer 3': {'M*': 2, 'M1': 0, 'M2': 1},
    'Layer 4': {'E*': 2, 'E1': 0, 'E2': 1},
    'Stock/Flow ID': {'F1': 0, 'F2': 1, 'F3': 2, 'F4': 3},
    'Substance_main_parent': {'P*': 2, 'P1': 0, 'P2': 1},
    'Unit': {'Mg': 0},
    'component': {'C*': 3, 'C1': 0, 'C2': 1, 'C3': 2},
    'element': {'E*': 2, 'E1': 0, 'E2': 1},
    'flow': {'F1': 0, 'F2': 1, 'F3': 2, 'F4': 3},
    'material': {'M*': 2, 'M1': 0, 'M2': 1},
    'parameterCode': {'c-p': 0, 'e-c': 2, 'e-m': 1, 'm-p': 3},
    'process': {'T1': 0, 'T2': 1},
    'product': {'P*': 2, 'P1': 0, 'P2': 1}}


---

# Linear equations


In [22]:
model.lneqs

<432x432 sparse matrix of type '<class 'numpy.float64'>'
	with 207 stored elements in Compressed Sparse Row format>

---

# Constant terms


In [23]:
model.y

<432x1 sparse array of type '<class 'numpy.int64'>'
	with 2 stored elements in Compressed Sparse Column format>

---

# Solver


In [24]:
#with fewer layers

solution = model.solve(aggregate=True, pivot=True)
solution.fillna("")

,flow,F1,F2,F3,F4
layer,key,,,,
product,P1,1000.00,,,
product,P2,700.00,,,
component,C1,551.00,86.58,,
component,C2,639.00,45.16,34.38,
component,C3,510.00,,12.80,
material,M1,884.00,244.40,166.40,
material,M2,816.00,81.60,240.00,
element,E1,1186.64,204.68,351.68,67.67
element,E2,513.36,121.32,54.72,30.86


In [30]:
#with extra layers

solution = model.solve(aggregate=True, pivot=True)
solution.fillna("")

,flow,F1,F2,F3,F4,F6,F5,F7,F8,F9
layer,key,,,,,,,,,
product,P1,1000.00,,,,,,,,
product,P2,700.00,,,,,,,,
component,C1,551.00,86.58,,,,,,,
component,C2,639.00,51.57,39.40,10.31,17.32,,,,
component,C3,510.00,,13.04,,1.83,,,,
material,M1,920.43,84.53,35.28,7.90,13.72,21.51,4.56,,
material,M2,779.57,53.62,17.16,2.41,5.42,19.33,4.99,,
element,E1,676.76,79.40,14.76,3.39,5.66,24.33,2.42,2.32,
element,E2,1023.24,58.75,37.68,6.93,13.48,16.51,7.14,1.19,


In [ ]:
#with 5 extra initial flows 

solution = model.solve(aggregate=True, pivot=True)
solution.fillna("")

,flow,WEEE_categ_mechRec1,WEEE_categ_mechRec2,WEEE_collected,WEEE_generatedComplementaryExported,WEEE_generatedComplementaryMSW,WEEE_generatedComplementaryMetalScrap,WEEE_generatedComplementaryUndocumented,WEEE_generatedDedicated0,WEEE_generatedDedicated1,WEEE_generatedDedicated2,...,WEEE_wasteBinLandfill,WEEE_2RM_mechRec1Smelter_other,WEEE_mechRec1_mechRec2,WEEE_2RM_mechRec2Smelter_other,WEEE_mechRec2_chem,WEEE_mechRec2_thermal,WEEE_2RM_chem2Smelter_other,WEEE_2RM_thermal2Smelter_other,WEEE_chem_thermal,WEEE_thermal_chem
layer,key,,,,,,,,,,,,,,,,,,,,,
product,WEEE_Cat1,1612592.45,2324030.29,4742918.96,677559.85,4658223.98,1778594.61,4150054.09,1693899.63,1693899.63,1693899.63,...,1676960.63,,,,,,,,,
product,WEEE_Cat2,499525.58,83254.26,925047.38,616698.25,2672359.09,1644528.67,2364009.96,1027830.42,1027830.42,1027830.42,...,454301.04,,,,,,,,,
product,WEEE_Cat3,37466.63,25289.97,93666.56,104686.16,314058.48,82646.97,126725.35,110195.96,110195.96,110195.96,...,91076.96,,,,,,,,,
product,WEEE_Cat4a,878888.89,2636666.66,6277777.75,2092592.58,8602880.62,10462962.92,8602880.62,4650205.74,4650205.74,4650205.74,...,2494835.38,,,,,,,,,
product,WEEE_Cat4b,21739.54,149459.35,271744.28,242366.52,212988.76,403944.20,183611.00,146888.80,146888.80,146888.80,...,44727.64,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
element,Pd,20689.13,38451.87,92931.38,106490.72,160820.17,182351.79,141771.16,81649.37,81649.37,81649.37,...,46857.79,1515.16,1068.87,744.78,,,,,,
element,Sc,37642.88,74734.88,184113.20,254194.44,353783.34,403377.25,289634.17,175588.23,175588.23,175588.23,...,105443.44,3185.29,2137.97,728.54,,,,,,
element,Sm,151148.47,287395.26,655505.88,611702.39,1020930.02,1071805.00,881372.70,499460.71,499460.71,499460.71,...,316540.46,4917.86,4917.12,18377.51,22.82,15.04,,,,


In [69]:
#with 10 extra initial flows
solution = model.solve(aggregate=True, pivot=True)
solution.fillna("")



RuntimeError: SUPERLU_MALLOC fails for buf in intCalloc() at line 172 in file ../scipy/sparse/linalg/_dsolve/SuperLU/SRC/memory.c


In [18]:
model.solve(aggregate=True, pivot=False)

,flow,layer,key,data
0,F1,product,P1,1000.00
1,F1,product,P2,700.00
0,F1,component,C1,551.00
1,F1,component,C2,639.00
2,F1,component,C3,510.00
3,F2,component,C1,86.58
4,F2,component,C2,51.57
5,F3,component,C2,39.40
6,F3,component,C3,13.04
7,F4,component,C2,10.31


In [19]:
model.solve(aggregate=False, pivot=True).fillna("")

flow,product,component,material,F1,F2,F3,F4,F5,F6,F7
0,P1,,,1000.00,,,,,,
1,P1,C1,,250.00,62.50,,,,,
2,P1,C1,M1,130.00,32.50,,,12.68,,
3,P1,C1,M2,120.00,30.00,,,9.60,,
4,P1,C2,,590.00,27.76,26.86,5.55,,11.24,
5,P1,C2,M1,389.40,18.32,17.73,3.66,1.83,7.42,2.08
6,P1,C2,M2,200.60,9.44,9.13,1.89,4.72,3.82,2.04
7,P1,C3,,160.00,,13.04,,,1.83,
8,P1,C3,M1,78.40,,6.39,,,0.89,0.83
9,P1,C3,M2,81.60,,6.65,,,0.93,2.59


In [47]:
model.solve(aggregate=False, pivot=False).fillna("")

,flow,product,component,material,element,data
0,WEEE_generatedComplementaryExported,WEEE_Cat1,,,,135511.97
1,WEEE_generatedComplementaryExported,WEEE_Cat1,ComponentShadowWEEE,,,1162.72
2,WEEE_generatedComplementaryExported,WEEE_Cat1,ComponentShadowWEEE,AlAndAlAlloys,,79.03
3,WEEE_generatedComplementaryExported,WEEE_Cat1,ComponentShadowWEEE,AlAndAlAlloys,Ag,11.68
4,WEEE_generatedComplementaryExported,WEEE_Cat1,ComponentShadowWEEE,AlAndAlAlloys,As,0.21
...,...,...,...,...,...,...
84461,WEEE_categ_mechRec1,WEEE_Cat6,wheel,polymerPA6GF30,Ag,23.64
84462,WEEE_categ_mechRec1,WEEE_Cat6,wheel,polymerPPTF20,,122.68
84463,WEEE_categ_mechRec1,WEEE_Cat6,wheel,polymerPPTF20,Ag,122.68
84464,WEEE_categ_mechRec1,WEEE_Cat6,wheel,polymerPPTF30,,114.06


---

# Performance


In [16]:
%%prun -l 20 -s cumulative

model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

         131892 function calls (129011 primitive calls) in 0.507 seconds

   Ordered by: cumulative time
   List reduced from 1309 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.506    0.506 <string>:1(<module>)
        1    0.000    0.000    0.506    0.506 <string>:2(__init__)
        1    0.000    0.000    0.506    0.506 recovery_model.py:37(__post_init__)
        1    0.003    0.003    0.373    0.373 recovery_model.py:219(read_tcs)
        5    0.002    0.000    0.123    0.025 recovery_model.py:584(encode_label)
        1    0.000    0.000    0.081    0.081 recovery_model.py:132(read_composition)
        4    0.000    0.000    0.077    0.019 readers.py:868(read_csv)
      369    0.008    0.000    0.068    0.000 construction.py:517(sanitize_array)
  214/208    0.008    0.000    0.068    0.000 series.py:389(__init__)
      128    0.005    0.000    0.065    0.001 base.py:475(__new__)
      189    0

In [17]:
cProfile.run(
    "RecoveryModel(name=folder, metadata=metadata, composition=composition, inputs=inputs, tcs=tcs, layer_names=layer_names, save_intermediary_steps=False, save_duplicates=False)",
    "performance/RecoveryModel.pstats",
)

cProfile.run(
    "model.solve(aggregate=False, pivot=True)",
    "performance/Solver.pstats",
)

# # ! then go into results/performances/ and run the following commands
# # (-n 2 means cutoff at 2%)
# gprof2dot -f pstats -n 1 RecoveryModel.pstats | dot -Tpng -o RecoveryModel.png
# gprof2dot -f pstats -n 1 Solver.pstats | dot -Tpng -o Solver.png

In [18]:
cProfile.run(
    "RecoveryModel(name=folder, metadata=metadata, composition=composition, inputs=inputs, tcs=tcs, layer_names=layer_names, save_intermediary_steps=False, save_duplicates=False)",
    "performance/RecoveryModel.prof",
)

cProfile.run(
    "model.solve(aggregate=False, pivot=True)",
    "performance/Solver.prof",
)

# # ! then go into results/performances/ and run the following commands
# snakeviz RecoveryModel.prof
# snakeviz Solver.prof